# Proyecto de Big Data para la Gestion Presupuestal

## Caso: Fundación Unibán y sus unidades de negocio

**Objetivo de la presentacion:** exponer la evolucion de un modelo analitico basado en Power Query y Power Pivot hacia una solucion moderna con Databricks, arquitectura medallion y una aplicacion web multiusuario.

**Alcance de esta entrega:** integracion y analisis de presupuesto aprobado y ejecucion para Fundación Unibán y sus principales unidades de negocio.

**Cobertura historica validada:** 2021 a 2025 en presupuesto y ejecucion.

**Fecha:** 19 de mayo de 2026


## 1. Resumen ejecutivo

- El proceso original ya contaba con una base analitica valiosa construida en Power Query, con transformaciones tipo medallion, y en Power Pivot, con tablas dinamicas y medidas para la presentacion.
- El principal reto era llevar esa logica a una plataforma mas reproducible, auditable y multiusuario.
- Se construyo un pipeline en Python que carga las fuentes a Databricks Student Edition usando la API SQL Statements.
- La informacion se organizo en Bronze, Silver y Gold para separar ingestion, normalizacion y consumo.
- Se habilito una aplicacion web para consultar presupuesto y ejecucion con filtros por año, unidad de negocio y clase financiera.

**Idea central:** conservar la logica financiera del modelo original, pero llevarla a una arquitectura de datos mas robusta, escalable y presentable academica y operacionalmente.


## 2. Contexto del problema y fuentes de informacion

### Contexto del modelo original
- La lectura financiera institucional se apoyaba en Power Query para la extraccion y transformacion de datos.
- Power Pivot, tablas dinamicas y medidas permitian consolidar la presentacion de resultados para diferentes unidades de negocio.
- El reto no era la ausencia de analitica, sino la dependencia de archivos locales, la actualizacion manual y la trazabilidad limitada del proceso completo.

### Fuentes integradas en esta entrega
- `2.Bases de datos/1. Ejecucion presupuestal`
- `2.Bases de datos/2. Presupuesto aprobado`
- `3.Consolidacion/A.Consolidacion`, especialmente la hoja `Datos`, como fuente de homologacion de centros de costo y relacion de rubros.

### Alcance temporal validado
- Se cargaron y validaron datos de presupuesto y ejecucion desde 2021 hasta 2025.
- Esto permite comparar comportamiento historico, continuidad del modelo y capacidad de evolucion hacia un entorno de datos moderno.


## 3. Arquitectura general

![Arquitectura general](assets/arquitectura_bigdata.svg)

### Lectura general de la arquitectura
- Los archivos institucionales siguen siendo la fuente de origen, pero ya no son el punto final del analisis.
- Python se utiliza como capa de ingestión y transformacion controlada.
- Databricks se utiliza como plataforma de almacenamiento y consulta analitica.
- La aplicacion web y el notebook funcionan como capas de consumo y comunicacion del resultado.


## 4. Pipeline medallion

![Pipeline medallion](assets/pipeline_medallion.svg)

### Significado de cada capa
- **Bronze:** inventario de archivos y trazabilidad de datos de origen.
- **Silver:** normalizacion de negocio para presupuesto y ejecucion.
- **Gold:** hechos, dimensiones y vistas de consumo para analitica y aplicacion.

### Estandarizacion apoyada en tablas maestras
La normalizacion del modelo se apoyo en la hoja `Datos` de los libros de consolidacion, especialmente en las tablas:
- `Rubros_Ingresos`
- `Rubros_Gastos`
- `HomologacionCeco`
- `Clasificacion_ceco`

---

### Evidencia real: catalogo desplegado en Databricks

Las siguientes capturas muestran el catalogo `presupuesto` tal como quedo en Databricks despues de ejecutar el pipeline, con los esquemas Bronze, Silver y Gold visibles en el Catalog Explorer.

**Catalogo presupuesto con esquemas Bronze, Silver y Gold**

![Catalogo Databricks](assets/databricks_catalogo.png)

**Tablas y vistas desplegadas en la arquitectura medallion**

![Detalle Gold, Bronze y Silver](assets/databricks_medallion_tables.png)


## 5. Aplicacion web de consulta

![MVP web](assets/app_mvp.svg)

### Capacidades de la aplicacion
- Autenticacion local para acceso controlado.
- Seleccion independiente de año de presupuesto y año de ejecucion.
- Filtros por unidad de negocio, clase financiera y detalle.
- Exportacion de resultados a CSV.
- Presentacion comparativa entre presupuesto y ejecucion con granularidad analitica util para la toma de decisiones.

---

### Capturas del MVP en produccion

**Libro Programas - resumen ejecutivo de marzo 2026**

![MVP Programas - resumen](assets/app_mvp_programas_resumen.png)

---

![MVP Programas - detalle](assets/app_mvp_programas_detalle.png)

---

![MVP Programas - accesos rapidos](assets/app_mvp_programas_accesos.png)


## 6. Resultados cuantitativos validados

| Indicador | Valor |
| --- | ---: |
| Filas de presupuesto normalizadas | 35,821 |
| Filas de ejecucion normalizadas | 342,133 |
| Centros de costo observados | 141 |
| Conceptos financieros en Gold | 137 |
| Registros de ejecucion clasificados con reglas de negocio | 341,633 |
| Cobertura de clasificacion automatizada | 99.85 % |
| Cobertura historica validada | 2021 a 2025 |

### Lectura academica del resultado
- La mayor parte del historico institucional quedo integrada y clasificada automaticamente.
- La arquitectura permite comparar presupuesto y ejecucion con una estructura consistente para varias unidades de negocio.
- Los casos residuales quedan identificados para auditoria posterior, en lugar de perderse dentro de la hoja de calculo.

### Impacto del proyecto
El resultado no es solo una migracion tecnica. Es una reorganizacion del proceso analitico institucional para hacerlo mas trazable, mas estable y mas facil de presentar y reutilizar.


In [ ]:
project_metrics = {
    "budget_rows": 35821,
    "execution_rows": 342133,
    "observed_cost_centers": 141,
    "financial_concepts": 137,
    "rule_classified_execution_rows": 341633,
    "automated_classification_coverage_pct": round(341633 / 342133 * 100, 2),
    "validated_years": list(range(2021, 2026)),
}

project_metrics


## 7. Demostracion local del backend

La siguiente celda ejecuta el mismo backend de la aplicacion para construir un reporte real.
Si el entorno local tiene credenciales configuradas en `.env`, deberia devolver el detalle de `Programas` para gastos en 2025.


In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    from config import load_settings
    from report_service import build_report

    settings = load_settings(project_root)
    payload = build_report(settings, 2025, 2025, 12, "EXPENSE", "Programas")
    result = payload["selected_classification"], payload["detail_rows"][:10]
    print("Conexion exitosa. Clasificacion seleccionada:", result[0])
    print(f"Primeras {len(result[1])} filas del detalle:")
    result
except FileNotFoundError:
    print("INFO: Archivo .env no encontrado en este entorno.")
    print("La demo requiere credenciales de Databricks configuradas localmente.")
    print("En produccion esta celda retorna el detalle de Programas para gastos 2025.")
except Exception as e:
    print(f"INFO: Demo no disponible en este entorno ({type(e).__name__}).")
    print("La demo requiere acceso activo al SQL Warehouse de Databricks.")
    print("Resultado esperado: clasificacion y primeras 10 filas del detalle de Programas 2025.")


## 8. Como presento el aporte de este trabajo

### Lo que construi
En este proyecto tome un proceso financiero que ya existia en Excel, Power Query y Power Pivot, y lo lleve a una arquitectura mas ordenada en Databricks para que no dependiera solo de archivos locales ni de actualizaciones manuales.

### Lo que deje resuelto
- Integre presupuesto aprobado, ejecucion y tablas maestras reales de la hoja `Datos`.
- Organice la informacion en capas Bronze, Silver y Gold para separar ingestion, normalizacion y consumo.
- Deje una capa Gold lista para consultas analiticas y para alimentar la aplicacion web.
- Valide informacion historica entre 2021 y 2025 para Fundacion Uniban y sus unidades de negocio.

### Por que lo ubico en Big Data
Yo lo sustento dentro de Big Data no solo por la cantidad de registros, sino por la integracion de varias fuentes, la estandarizacion de reglas de negocio, la trazabilidad del pipeline y la construccion de una capa analitica reutilizable.

### Salida real para los usuarios
En lugar de dejar esto como una proyeccion hacia Power BI o Databricks Dashboards, mi enfoque fue desarrollar una plataforma web multiusuario, con apoyo de ClaudeCode y Codex, para que los stakeholders puedan revisar presupuesto y ejecucion con filtros utiles para el seguimiento financiero.

## Cierre

En este trabajo quise mostrar que un proceso financiero institucional puede pasar de archivos locales y transformaciones manuales a una solucion mas trazable, reutilizable y preparada para crecer, sin perder la logica del negocio que ya existia.

Lo mas importante para mi fue dejar funcionando un pipeline con estructura Bronze, Silver y Gold, soportado en Databricks, y al mismo tiempo construir una plataforma web multiusuario para consultar presupuesto y ejecucion de una forma mas clara para los stakeholders.

Si mas adelante hace falta conectar Power BI o Databricks Dashboards, la capa Gold ya deja abierta esa posibilidad. Pero en esta entrega mi resultado principal no es una proyeccion futura, sino la plataforma que ya estoy desarrollando y el modelo de datos que la alimenta.